In [6]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# Load in Account A data
df = pd.read_csv("account_a.csv")

# Convert Datetime variable so it is parsed correctly
df["Datetime"] = pd.to_datetime(df["Datetime"], format="mixed", errors="coerce")

# Extract date and hour
df["date_only"] = df["Datetime"].dt.date
df["hour"] = df["Datetime"].dt.hour

# First 2 weeks (14 days) only
time = sorted(df["date_only"].unique())[:14]
df_14 = df[df["date_only"].isin(time)].copy()

# Map days and hours indexes
day_map = {day: i + 1 for i, day in enumerate(time)}
df_14["d"] = df_14["date_only"].map(day_map)
df_14["t"] = df_14["hour"]

# Calculate Price and Quantity
hourly = (
    df_14.groupby(["d", "t"], as_index=False)
    .agg({
        "Quantity": "sum",
        "Settlement.Point.Price.KWH": "mean"
    })
)

Price = {(row["d"], row["t"]): row["Settlement.Point.Price.KWH"] for _, row in hourly.iterrows()}
Quantity = {(row["d"], row["t"]): row["Quantity"] for _, row in hourly.iterrows()}

# Create set indexes
DT = sorted(list(Quantity.keys()))  

# Max batteries for 2 week period
B_max = 30

# Battery cost
C_battery = 14900

# Charging/Discharging rate per battery per hour
r_ch = 5
r_dis = 11.5

m = gp.Model("battery_model_account_a")

# Decision Variables
x_A = m.addVar(vtype=GRB.INTEGER, lb=0, ub=B_max)
e = m.addVars(range(1, B_max + 1), vtype=GRB.BINARY)
y = m.addVars(range(1, B_max + 1), DT, vtype=GRB.BINARY)
charge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
discharge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
cs = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
charge_b = m.addVars(range(1, B_max+1), DT, lb=0, vtype=GRB.CONTINUOUS)
discharge_b = m.addVars(range(1, B_max+1), DT, lb=0, vtype=GRB.CONTINUOUS)


# Constraints
# sum_b e_A,b = x_A
m.addConstr(
    gp.quicksum(e[b] for b in range(1, B_max + 1)) == x_A,
    name="num_active_batteries"
)

# y_b,A,t,d <= e_A,b
for b in range(1, B_max + 1):
    for d, t in DT:
        m.addConstr(y[b, d, t] <= e[b])

# g_{A,t,d} = Quantity_{A,t,d} 
for d, t in DT: 
    m.addConstr(discharge[d, t] == Quantity[d, t])

    
# charging state capacity constraints
for d, t in DT:
    m.addConstr(cs[d, t] <= 13.5 * x_A)
    m.addConstr(cs[d, t] >= 2.7 * x_A)


# charging/discharging rate limits
for d, t in DT:
    m.addConstr(charge[d, t] <= r_ch * x_A)
    m.addConstr(discharge[d, t] <= r_dis * x_A)

# Charging state across time
DT_sorted = sorted(DT)   

# Charging state constraints
for i, (d, t) in enumerate(DT_sorted):
    if i == 0:
        m.addConstr(
            cs[d, t] == 13.5 * x_A + charge[d, t] - discharge[d, t]
        )
    else:
        prev_d, prev_t = DT_sorted[i - 1]
        m.addConstr(
            cs[d, t] == cs[prev_d, prev_t] + charge[d, t] - discharge[d, t]
        )

# Battery charging constraint
for b in range(1, B_max+1):
    for d, t in DT:
        m.addConstr(
            charge_b[b, d, t] <= r_ch * y[b, d, t],
            name=f"charge_mode_b{b}_d{d}_t{t}"
        )

# Battery discharging constraint
for b in range(1, B_max+1):
    for d, t in DT:
        m.addConstr(
            discharge_b[b, d, t] <= r_dis * (e[b] - y[b, d, t]),
            name=f"discharge_mode_b{b}_d{d}_t{t}"
        )

# Sum of all charge and discharge for each battery must equal the totals
for d, t in DT:
    m.addConstr(
        charge[d, t] == gp.quicksum(charge_b[b, d, t] for b in range(1, B_max+1)),
        name=f"charge_total_d{d}_t{t}"
    )
    
    m.addConstr(
        discharge[d, t] == gp.quicksum(discharge_b[b, d, t] for b in range(1, B_max+1)),
        name=f"discharge_total_d{d}_t{t}"
    )


# Objective Function
energy_cost = gp.quicksum(
    charge[d, t] * Price[d, t]
    for d, t in DT
)

battery_cost = x_A * C_battery

m.setObjective(energy_cost + battery_cost, GRB.MINIMIZE)

# Optimize
m.optimize()

# Print results
if m.status == GRB.OPTIMAL:
    print("\nOptimal Solution")
    print(f"x_A = {x_A.X}")
    print(f"Energy cost = {energy_cost.getValue():,.4f}")
    print(f"Battery cost = {battery_cost.getValue():,.4f}")
    print(f"Total Cost = {m.objVal:,.4f}")

    # Display charge/discharge values for some iterations just to check
    print("\nSample hourly values:")
    for d, t in DT_sorted[:600]:
        print(
            f"(d={d}, t={t}) | charge = {charge[d,t].X:.4f} kWh, "
            f"discharge = {discharge[d,t].X:.4f} kWh, charge_state = {cs[d,t].X:.4f} kWh, price = ${Price[(d,t)]:.4f}"
        )
else:
    print("No optimal solution found.")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 155U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 22639 rows, 21514 columns and 65866 nonzeros
Model fingerprint: 0x4a8a9661
Variable types: 14553 continuous, 6961 integer (6960 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [6e-03, 1e+04]
  Bounds range     [1e+00, 3e+01]
  RHS range        [1e+00, 5e+00]
Presolve removed 696 rows and 462 columns
Presolve time: 0.22s
Presolved: 21943 rows, 21052 columns, 64473 nonzeros
Variable types: 14091 continuous, 6961 integer (6960 binary)
Found heuristic solution: objective 447017.08350
Found heuristic solution: objective 447016.33947

Root relaxation: objective 2.272915e+04, 18628 iterations, 1.70 seconds (0.91 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Ex

In [7]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# Load in Account A data
df = pd.read_csv("account_b.csv")

# Convert Datetime variable so it is parsed correctly
df["Datetime"] = pd.to_datetime(df["Datetime"], format="mixed", errors="coerce")

# Extract date and hour
df["date_only"] = df["Datetime"].dt.date
df["hour"] = df["Datetime"].dt.hour

# First 2 weeks (14 days) only
time = sorted(df["date_only"].unique())[:14]
df_14 = df[df["date_only"].isin(time)].copy()

# Map days and hours indexes
day_map = {day: i + 1 for i, day in enumerate(time)}
df_14["d"] = df_14["date_only"].map(day_map)
df_14["t"] = df_14["hour"]

# Calculate Price and Quantity
hourly = (
    df_14.groupby(["d", "t"], as_index=False)
    .agg({
        "Quantity": "sum",
        "Settlement.Point.Price.KWH": "mean"
    })
)

Price = {(row["d"], row["t"]): row["Settlement.Point.Price.KWH"] for _, row in hourly.iterrows()}
Quantity = {(row["d"], row["t"]): row["Quantity"] for _, row in hourly.iterrows()}

# Create set indexes
DT = sorted(list(Quantity.keys()))  

# Max batteries for 2 week period
B_max = 110

# Battery cost
C_battery = 14900

# Charging/Discharging rate per battery per hour
r_ch = 5
r_dis = 11.5

m = gp.Model("battery_model_account_b")

# Decision Variables
x_B = m.addVar(vtype=GRB.INTEGER, lb=0, ub=B_max)
e = m.addVars(range(1, B_max + 1), vtype=GRB.BINARY)
y = m.addVars(range(1, B_max + 1), DT, vtype=GRB.BINARY)
charge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
discharge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
cs = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
charge_b = m.addVars(range(1, B_max+1), DT, lb=0, vtype=GRB.CONTINUOUS)
discharge_b = m.addVars(range(1, B_max+1), DT, lb=0, vtype=GRB.CONTINUOUS)

# Constraints
# sum_b e_A,b = x_B
m.addConstr(
    gp.quicksum(e[b] for b in range(1, B_max + 1)) == x_B,
    name="num_active_batteries"
)

# y_b,A,t,d <= e_A,b
for b in range(1, B_max + 1):
    for d, t in DT:
        m.addConstr(y[b, d, t] <= e[b])

# g_{A,t,d} = Quantity_{A,t,d} 
for d, t in DT: 
    m.addConstr(discharge[d, t] == Quantity[d, t])
    
# charging state capacity constraints
for d, t in DT:
    m.addConstr(cs[d, t] <= 13.5 * x_B)
    m.addConstr(cs[d, t] >= 2.7 * x_B)


# charging/discharging rate limits
for d, t in DT:
    m.addConstr(charge[d, t] <= r_ch * x_B)
    m.addConstr(discharge[d, t] <= r_dis * x_B)

# Charging state across time
DT_sorted = sorted(DT)   

# Charging state constraints
for i, (d, t) in enumerate(DT_sorted):
    if i == 0:
        m.addConstr(
            cs[d, t] == 13.5 * x_B + charge[d, t] - discharge[d, t]
        )
    else:
        prev_d, prev_t = DT_sorted[i - 1]
        m.addConstr(
            cs[d, t] == cs[prev_d, prev_t] + charge[d, t] - discharge[d, t]
        )

# Battery charging constraint
for b in range(1, B_max+1):
    for d, t in DT:
        m.addConstr(
            charge_b[b, d, t] <= r_ch * y[b, d, t],
            name=f"charge_mode_b{b}_d{d}_t{t}"
        )

# Battery discharging constraint
for b in range(1, B_max+1):
    for d, t in DT:
        m.addConstr(
            discharge_b[b, d, t] <= r_dis * (e[b] - y[b, d, t]),
            name=f"discharge_mode_b{b}_d{d}_t{t}"
        )

# Sum of all charge and discharge for each battery must equal the totals
for d, t in DT:
    m.addConstr(
        charge[d, t] == gp.quicksum(charge_b[b, d, t] for b in range(1, B_max+1)),
        name=f"charge_total_d{d}_t{t}"
    )
    
    m.addConstr(
        discharge[d, t] == gp.quicksum(discharge_b[b, d, t] for b in range(1, B_max+1)),
        name=f"discharge_total_d{d}_t{t}"
    )


# Objective Function
energy_cost = gp.quicksum(
    charge[d, t] * Price[d, t]
    for d, t in DT
)

battery_cost = x_B * C_battery

m.setObjective(energy_cost + battery_cost, GRB.MINIMIZE)

# Optimize
m.optimize()

# Print results
if m.status == GRB.OPTIMAL:
    print("\nOptimal Solution")
    print(f"x_B = {x_B.X}")
    print(f"Energy cost = {energy_cost.getValue():,.4f}")
    print(f"Battery cost = {battery_cost.getValue():,.4f}")
    print(f"Total Cost = {m.objVal:,.4f}")

    # Display charge/discharge values for some iterations just to check
    print("\nSample hourly values:")
    for d, t in DT_sorted[:600]:
        print(
            f"(d={d}, t={t}) | charge = {charge[d,t].X:.4f} kWh, "
            f"discharge = {discharge[d,t].X:.4f} kWh, charge_state = {cs[d,t].X:.4f} kWh, price = ${Price[(d,t)]:.4f}"
        )
else:
    print("No optimal solution found.")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 155U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 78079 rows, 77034 columns and 232266 nonzeros
Model fingerprint: 0x1651188f
Variable types: 51513 continuous, 25521 integer (25520 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [6e-03, 1e+04]
  Bounds range     [1e+00, 1e+02]
  RHS range        [9e+01, 3e+02]
Presolve removed 9046 rows and 462 columns
Presolve time: 0.92s
Presolved: 69033 rows, 76572 columns, 214173 nonzeros
Variable types: 51051 continuous, 25521 integer (25520 binary)
Found heuristic solution: objective 1640821.6724
Found heuristic solution: objective 1640820.9606
Deterministic concurrent LP optimizer: primal and dual simplex
Showing primal log only...

Concurrent spin time: 0.01s

Solved with dual simplex

Use crossover t